In [ ]:
import numpy as np
from collections import Counter

# --- Parcours de Jordan ---
# [tirs_cadres, tirs_cadres_concedes]
jordan_matches = [
    [5, 1],   # vs Arabie Saoudite (1-0)
    [4, 3],   # vs Irak (1-0)
    [6, 2],   # vs UAE (2-1)
    [6, 2],   # vs Koweit (3-1)
    [6, 7]    # vs Egypte (3-0)
]

# --- Parcours du Maroc ---
maroc_matches = [
    [7, 2],   # vs UAE (3-0)
    [8, 1],   # vs Syrie (1-0)
    [5, 3],   # vs Comoros (3-1)
    [0, 1],   # vs Oman (0-0)
    [2, 0]    # vs Arabie Saoudite (1-0)
]

def calc_xg_proxy(matches):
    tirs_cadres = [m[0] for m in matches]
    tirs_cadres_concedes = [m[1] for m in matches]
    attaque = np.mean(tirs_cadres)
    defense = np.mean(tirs_cadres_concedes)
    return attaque, defense

att_jord, def_jord = calc_xg_proxy(jordan_matches)
att_maroc, def_maroc = calc_xg_proxy(maroc_matches)

# --- Buts attendus (lambda) ---
# On normalise par un facteur pour convertir tirs cadrés -> but attendu
#factor = 0.15  # approx conversion tirs cadrés -> xG
# factor_mar = 0.3307
# factor_jord = 0.3566
factor_mar = 0.3636 / 2
factor_jord = 0.3703 / 2
lambda_maroc = att_maroc * def_jord * factor_mar
lambda_jord = att_jord * def_maroc * factor_jord

print(f"Buts attendus - Maroc: {lambda_maroc:.2f}, Jordanie: {lambda_jord:.2f}")

# --- Simulation Monte Carlo ---
simulations = 1000000
results = []

for _ in range(simulations):
    buts_maroc = np.random.poisson(lambda_maroc)
    buts_jord = np.random.poisson(lambda_jord)

    # Prolongation si match nul
    if buts_maroc == buts_jord:
        buts_maroc += np.random.poisson(0.3)
        buts_jord += np.random.poisson(0.3)

    if buts_maroc > buts_jord:
        results.append("Maroc")
    elif buts_jord > buts_maroc:
        results.append("Jordanie")
    else:
        results.append("Nul")

counts = Counter(results)
total = sum(counts.values())

print("\nProbabilités du match :")
for team, count in counts.items():
    print(f"{team}: {count/total:.2%}")

# --- Score le plus probable ---
score_counts = Counter()
for _ in range(simulations):
    buts_maroc = np.random.poisson(lambda_maroc)
    buts_jord = np.random.poisson(lambda_jord)
    if buts_maroc == buts_jord:
        buts_maroc += np.random.poisson(0.3)
        buts_jord += np.random.poisson(0.3)
    score_counts[(buts_maroc, buts_jord)] += 1

score_prob = score_counts.most_common(1)[0]
print(f"\nScore le plus probable : Maroc {score_prob[0][0]} - Jordanie {score_prob[0][1]}")


Buts attendus - Maroc: 2.40, Jordanie: 1.40

Probabilités du match :
Maroc: 63.44%
Jordanie: 25.15%
Nul: 11.41%

Score le plus probable : Maroc 2 - Jordanie 1


In [ ]:
import numpy as np
import pandas as pd

# --- Parcours de Jordan et Maroc (tirs cadrés offensifs et concédés) ---
# --- Parcours de Jordan ---
# [tirs_cadres, tirs_cadres_concedes]
jordan_matches = [
    [5, 1],   # vs Arabie Saoudite (1-0)
    [4, 3],   # vs Irak (1-0)
    [6, 2],   # vs UAE (2-1)
    [6, 2],   # vs Koweit (3-1)
    [6, 7]    # vs Egypte (3-0)
]

# --- Parcours du Maroc ---
maroc_matches = [
    [7, 2],   # vs UAE (3-0)
    [8, 1],   # vs Syrie (1-0)
    [5, 3],   # vs Comoros (3-1)
    [0, 1],   # vs Oman (0-0)
    [2, 0]    # vs Arabie Saoudite (1-0)
]

def calc_xg_proxy(matches):
    tirs_cadres = [m[0] for m in matches]
    tirs_cadres_concedes = [m[1] for m in matches]
    attaque = np.mean(tirs_cadres)
    defense = np.mean(tirs_cadres_concedes)
    return attaque, defense

att_jord, def_jord = calc_xg_proxy(jordan_matches)
att_maroc, def_maroc = calc_xg_proxy(maroc_matches)

# --- Buts attendus ---
factor_mar = 0.3636 / 1.6
factor_jord = 0.3703 /1.6
# factor_mar = 0.3307
# factor_jord = 0.3566
lambda_maroc = att_maroc * def_jord * factor_mar
lambda_jord = att_jord * def_maroc * factor_jord

# --- Simulation Monte Carlo ---
simulations = 1000000
max_goals = 6
score_counts = np.zeros((max_goals, max_goals))
result_counts = {"Maroc":0, "Jordanie":0, "Nul":0}

for _ in range(simulations):
    buts_maroc = np.random.poisson(lambda_maroc)
    buts_jord = np.random.poisson(lambda_jord)

    # Prolongation légère si match nul
    if buts_maroc == buts_jord:
        buts_maroc += np.random.poisson(0.3)
        buts_jord += np.random.poisson(0.3)

    # Limiter à max_goals pour le tableau
    buts_maroc = min(buts_maroc, max_goals-1)
    buts_jord = min(buts_jord, max_goals-1)

    # Compter le score exact
    score_counts[buts_maroc, buts_jord] += 1

    # Compter le résultat du match
    if buts_maroc > buts_jord:
        result_counts["Maroc"] += 1
    elif buts_jord > buts_maroc:
        result_counts["Jordanie"] += 1
    else:
        result_counts["Nul"] += 1

# --- Probabilités ---
prob_maroc = result_counts["Maroc"]/simulations
prob_jord = result_counts["Jordanie"]/simulations
prob_nul = result_counts["Nul"]/simulations

# --- Score le plus probable ---
i, j = np.unravel_index(score_counts.argmax(), score_counts.shape)

# --- Créer tableau complet des scores et probabilités ---
scores = [f"{i}-{j}" for i in range(max_goals) for j in range(max_goals)]
probs = [score_counts[i,j]/simulations for i in range(max_goals) for j in range(max_goals)]
df_scores = pd.DataFrame({"Score": scores, "Probabilité": probs})

# Trier par probabilité décroissante
df_scores = df_scores.sort_values(by="Probabilité", ascending=False).reset_index(drop=True)

# --- Tableau final résumé ---
final_table = pd.DataFrame({
    "Buts attendus Maroc": [lambda_maroc],
    "Buts attendus Jordanie": [lambda_jord],
    "Prob. Maroc gagne": [prob_maroc],
    "Prob. Jordanie gagne": [prob_jord],
    "Prob. Nul": [prob_nul],
    "Score le plus probable": [f"{i}-{j}"]
})

# Affichage
print("=== Tableau résumé du match ===")
print(final_table)

print("\n=== Top 15 scores les plus probables ===")
print(df_scores.head(15))


=== Tableau résumé du match ===
   Buts attendus Maroc  Buts attendus Jordanie  Prob. Maroc gagne  \
0               2.9997                1.749667           0.661598   

   Prob. Jordanie gagne  Prob. Nul Score le plus probable  
0              0.235947   0.102455                    2-1  

=== Top 15 scores les plus probables ===
   Score  Probabilité
0    2-1     0.075797
1    3-2     0.070117
2    3-1     0.069229
3    5-1     0.056420
4    4-1     0.050940
5    5-2     0.049243
6    1-2     0.047321
7    4-2     0.046354
8    2-3     0.045025
9    2-0     0.039263
10   3-0     0.038802
11   2-2     0.034719
12   4-3     0.032411
13   5-0     0.032119
14   5-3     0.029576


In [ ]:
final_table

,Buts attendus Maroc,Buts attendus Jordanie,Prob. Maroc gagne,Prob. Jordanie gagne,Prob. Nul,Score le plus probable
0,2.9997,1.749667,0.661598,0.235947,0.102455,2-1


In [ ]:
df_scores

,Score,Probabilité
0,2-1,0.075797
1,3-2,0.070117
2,3-1,0.069229
3,5-1,0.056420
4,4-1,0.050940
5,5-2,0.049243
6,1-2,0.047321
7,4-2,0.046354
8,2-3,0.045025
9,2-0,0.039263
